In [0]:
%sql
-- Create the main catalog
CREATE CATALOG IF NOT EXISTS globalmart
COMMENT 'GlobalMart E-Commerce Databricks Catalog for Medallion Architecture';

-- Set active catalog
USE CATALOG globalmart;

-- Create Bronze Schema
CREATE SCHEMA IF NOT EXISTS globalmart.bronze
COMMENT 'Bronze Layer: Raw, append-only, immutable data directly ingested from source CSV files with added metadata.';

-- Create Silver Schema
CREATE SCHEMA IF NOT EXISTS globalmart.silver
COMMENT 'Silver Layer: Cleaned, deduplicated, validated, and normalized data providing a single source of truth.';

-- Create Gold Schema
CREATE SCHEMA IF NOT EXISTS globalmart.gold
COMMENT 'Gold Layer: Business-level aggregations, KPI metrics, and Dimensional Models (Star Schema) for analytics and reporting.';

-- Create Metadata / Operational Tracking Schema
CREATE SCHEMA IF NOT EXISTS globalmart.meta
COMMENT 'Metadata & Operational Layer: Stores pipeline execution logs, audit trails, and data quality check results.';

In [0]:
%sql
SHOW SCHEMAS IN globalmart;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS globalmart.meta.ingestion_log (
    file_name STRING,
    file_path STRING,
    table_name STRING,
    records_ingested LONG,
    ingested_at TIMESTAMP,
    status STRING
)
USING DELTA
COMMENT 'Operational metadata table tracking processed files to ensure idempotent pipeline runs.';

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

# Mapping source CSV files to Bronze Delta table names
dataset_files = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_order_payments",
    "olist_order_reviews_dataset.csv": "bronze_order_reviews",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers"
}

VOLUME_PATH = "/Volumes/workspace/data/raw_data/"

def ingest_file_idempotent(file_name: str, table_name: str) -> None:
    """
    Ingests a raw CSV file into the Bronze Delta table idempotently.
    Skips processing if the file entry exists in the metadata log table.
    """
    file_path = f"{VOLUME_PATH}{file_name}"
    
    # 1. Check if the file has already been processed successfully
    log_df = spark.table("globalmart.meta.ingestion_log")
    already_processed = log_df.filter(
        (col("file_name") == file_name) & (col("status") == "SUCCESS")
    ).count() > 0
    
    if already_processed:
        print(f"INFO: Skipping '{file_name}'. Already ingested into globalmart.bronze.{table_name}.")
        return
    
    print(f"INFO: Ingesting '{file_name}' into globalmart.bronze.{table_name}...")
    
    # 2. Read raw CSV file
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(file_path))
    
    # 3. Enrich dataset with operational metadata
    enriched_df = (df
                   .withColumn("_ingested_at", current_timestamp())
                   .withColumn("_source_file", lit(file_name)))
    
    record_count = enriched_df.count()
    
    # 4. Write data to Bronze layer as a Delta Lake table
    enriched_df.write.format("delta") \
              .mode("append") \
              .saveAsTable(f"globalmart.bronze.{table_name}")
    
    # 5. Record execution metadata in the logging table
    log_data = [(
        file_name,
        file_path,
        table_name,
        record_count,
        spark.sql("SELECT current_timestamp()").collect()[0][0],
        "SUCCESS"
    )]
    
    log_record_df = spark.createDataFrame(log_data, log_df.schema)
    log_record_df.write.format("delta").mode("append").saveAsTable("globalmart.meta.ingestion_log")
    
    print(f"SUCCESS: Ingested {record_count:,} records into globalmart.bronze.{table_name}.")

# Execute ingestion across all source files
for file_name, table_name in dataset_files.items():
    ingest_file_idempotent(file_name, table_name)

In [0]:
# Re-run ingestion framework on the orders file to test idempotency
ingest_file_idempotent("olist_orders_dataset.csv", "bronze_orders")

# Verify record count remains identical and no duplicate records were added
display(spark.sql("SELECT COUNT(*) AS total_records FROM globalmart.bronze.bronze_orders"))

In [0]:
%sql
SELECT 
    table_name,
    records_ingested AS record_count,
    (SELECT COUNT(*) 
     FROM system.information_schema.columns 
     WHERE table_catalog = 'globalmart' 
       AND table_schema = 'bronze' 
       AND table_name = l.table_name) AS column_count,
    status AS ingestion_status,
    ingested_at
FROM globalmart.meta.ingestion_log l
ORDER BY records_ingested DESC;

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Target directory for Auto Loader checkpointing
CHECKPOINT_PATH = "/Volumes/workspace/data/raw_data/_checkpoints/orders_autoloader"
VOLUME_PATH = "/Volumes/workspace/data/raw_data/"

# Define stream using Auto Loader
autoloader_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
    .load(VOLUME_PATH)
    .filter(col("_metadata.file_path").contains("olist_orders_dataset.csv"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

# Write using trigger AvailableNow for Community Edition compatibility
query = (
    autoloader_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable("globalmart.bronze.bronze_orders_autoloader")
)

# Wait for stream processing to complete
query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS initial_row_count 
FROM globalmart.bronze.bronze_orders_autoloader;

In [0]:
# Trigger Auto Loader a second time over the same directory
query_second_run = (
    autoloader_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable("globalmart.bronze.bronze_orders_autoloader")
)

query_second_run.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS second_run_row_count 
FROM globalmart.bronze.bronze_orders_autoloader;

In [0]:
from pyspark.sql.functions import col, collect_list, struct, count, sum as _sum, size

# Load bronze order payments table
payments_df = spark.table("globalmart.bronze.bronze_order_payments")

# Create nested representation: grouping by order_id
nested_payments_df = (
    payments_df.groupBy("order_id")
    .agg(
        _sum("payment_value").alias("total_payment_value"),
        count("payment_sequential").alias("num_payment_methods"),
        collect_list(
            struct(
                col("payment_sequential"),
                col("payment_type"),
                col("payment_installments"),
                col("payment_value")
            )
        ).alias("payment_details")
    )
)

# Save nested representation as a Delta table in Silver
nested_payments_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("globalmart.silver.silver_payments_nested")

print("SUCCESS: Nested payments table created.")

In [0]:
from pyspark.sql.functions import explode

# Read nested table and explode the payment_details struct array
flattened_payments_df = (
    spark.table("globalmart.silver.silver_payments_nested")
    .select(
        col("order_id"),
        col("total_payment_value"),
        col("num_payment_methods"),
        explode(col("payment_details")).alias("payment")
    )
    .select(
        col("order_id"),
        col("total_payment_value"),
        col("num_payment_methods"),
        col("payment.payment_sequential").alias("payment_sequential"),
        col("payment.payment_type").alias("payment_type"),
        col("payment.payment_installments").alias("payment_installments"),
        col("payment.payment_value").alias("payment_value")
    )
)

# Save flattened representation as a Delta table in Silver
flattened_payments_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("globalmart.silver.silver_payments_flattened")

print("SUCCESS: Flattened payments table created.")

In [0]:
# Extract metrics for verification
raw_count = spark.table("globalmart.bronze.bronze_order_payments").count()
distinct_orders_count = spark.table("globalmart.bronze.bronze_order_payments").select("order_id").distinct().count()
nested_count = spark.table("globalmart.silver.silver_payments_nested").count()
flattened_count = spark.table("globalmart.silver.silver_payments_flattened").count()

print(f"Raw Payments Row Count:           {raw_count:,}")
print(f"Distinct Order ID Count:          {distinct_orders_count:,}")
print(f"Nested Payments Row Count:        {nested_count:,}")
print(f"Flattened Payments Row Count:     {flattened_count:,}")

# Verification checks
assert nested_count == distinct_orders_count, "FAILED: Nested count does not match distinct orders count!"
assert flattened_count == raw_count, "FAILED: Flattened count does not match original raw count!"

print("\nVERIFICATION SUCCESSFUL: Nested and Flattened representations match required metrics perfectly.")

In [0]:
%sql
SELECT 
    MAX(num_payment_methods) AS max_payment_methods_on_single_order,
    COUNT(CASE WHEN num_payment_methods > 1 THEN 1 END) AS multi_payment_orders_count,
    COUNT(*) AS total_orders_count,
    ROUND((COUNT(CASE WHEN num_payment_methods > 1 THEN 1 END) * 100.0) / COUNT(*), 2) AS percentage_multi_payment_orders
FROM globalmart.silver.silver_payments_nested;

In [0]:
from pyspark.sql import functions as F

# Base volume path
VOLUME_PATH = "/Volumes/workspace/data/raw_data/"
raw_orders_path = f"{VOLUME_PATH}olist_orders_dataset.csv"

# Read raw orders dataset
orders_df = spark.read.option("header", "true").csv(raw_orders_path)

# Split data: Keep 80% as base records, 20% to evolve
base_orders_df, new_orders_raw = orders_df.randomSplit([0.8, 0.2], seed=42)

# Add two new categorical columns to the 20% sample batch
evolved_new_orders_df = new_orders_raw \
    .withColumn("order_priority", F.expr("CASE WHEN order_status = 'delivered' THEN 'MEDIUM' ELSE 'HIGH' END")) \
    .withColumn("fulfillment_channel", F.lit("DIRECT_WAREHOUSE"))

# Save evolved batch to volume
evolved_csv_path = f"{VOLUME_PATH}olist_orders_dataset_evolved.csv"
evolved_new_orders_df.write.mode("overwrite").option("header", "true").csv(evolved_csv_path)

print(f"SUCCESS: Evolved CSV generated at {evolved_csv_path}")

In [0]:
# Checkpoint and Schema locations
checkpoint_path = f"{VOLUME_PATH}_checkpoints/bronze_orders_evolved"
schema_path = f"{VOLUME_PATH}_schemas/bronze_orders_evolved"

# Stream read using Auto Loader
evolved_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header", "true")
    .load(evolved_csv_path)
)

# Write to bronze table using mergeSchema and availableNow trigger
query = (
    evolved_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("globalmart.bronze.bronze_orders")
)

query.awaitTermination()
print("SUCCESS: Evolved data re-ingested into globalmart.bronze.bronze_orders.")

In [0]:
orders_bronze_df = spark.table("globalmart.bronze.bronze_orders")

old_null_count = orders_bronze_df.filter(F.col("order_priority").isNull()).count()
new_populated_count = orders_bronze_df.filter(F.col("order_priority").isNotNull()).count()

print("--- Verification Deliverable ---")
print(f"Old Records (order_priority IS NULL):     {old_null_count:,}")
print(f"New Records (order_priority IS NOT NULL): {new_populated_count:,}")

# Sample preview
display(
    orders_bronze_df.select(
        "order_id", "order_status", "order_priority", "fulfillment_channel"
    ).limit(10)
)

In [0]:
# 1. Create Schema Violation Log Table in `meta` schema
spark.sql("""
CREATE TABLE IF NOT EXISTS globalmart.meta.schema_violation_log (
    table_name STRING,
    violation_type STRING,
    column_name STRING,
    expected_value STRING,
    actual_value STRING,
    log_timestamp TIMESTAMP
) USING DELTA;
""")

# 2. Define Schema Validator
def validate_schema(df, expected_schema_dict, table_name):
    """
    Compares DF schema against expected contract.
    Logs missing columns, extra columns, or type mismatches to globalmart.meta.schema_violation_log.
    """
    actual_schema = {field.name: field.dataType.simpleString() for field in df.schema}
    violations = []
    
    # Check Missing Columns and Type Mismatches
    for col_name, expected_type in expected_schema_dict.items():
        if col_name not in actual_schema:
            violations.append((table_name, "MISSING_COLUMN", col_name, expected_type, "NONE"))
        elif actual_schema[col_name] != expected_type:
            violations.append((table_name, "TYPE_MISMATCH", col_name, expected_type, actual_schema[col_name]))
            
    # Check Extra Columns
    for col_name, actual_type in actual_schema.items():
        # Exclude Auto Loader internal columns
        if col_name in ["_rescued_data", "_ingested_at", "_source_file"]:
            continue
        if col_name not in expected_schema_dict:
            violations.append((table_name, "EXTRA_COLUMN", col_name, "NONE", actual_type))
            
    # Log violations to Delta table
    if violations:
        violation_df = spark.createDataFrame(
            violations, 
            ["table_name", "violation_type", "column_name", "expected_value", "actual_value"]
        ).withColumn("log_timestamp", F.current_timestamp())
        
        violation_df.write.format("delta").mode("append").saveAsTable("globalmart.meta.schema_violation_log")
        print(f"LOGGED: {len(violations)} schema violation(s) recorded in globalmart.meta.schema_violation_log.")
    else:
        print(f"SUCCESS: Schema validation passed for '{table_name}'. No violations found.")

# 3. Expected Baseline Schema Contract
expected_orders_schema = {
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string",
    "order_purchase_timestamp": "string",
    "order_approved_at": "string",
    "order_delivered_carrier_date": "string",
    "order_delivered_customer_date": "string",
    "order_estimated_delivery_date": "string"
}

# Run validation on evolved table
validate_schema(orders_bronze_df, expected_orders_schema, "globalmart.bronze.bronze_orders")

# Query Violation Logs
display(spark.table("globalmart.meta.schema_violation_log"))